# ResNet18 Stratified 5-Fold Cross Validation

## Objective

The objective of this notebook is to evaluate the ResNet18 architecture using Stratified 5-Fold Cross Validation.

Compared with a simple train-test split, cross-validation provides a more reliable estimate of model performance by ensuring every image is used for both training and validation exactly once.

## Import Required Libraries

Import the libraries required for transfer learning, dataset handling, optimization, evaluation, and cross-validation.

In [ ]:
import os
import copy
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Image Preprocessing

Define the training and validation preprocessing pipelines using ImageNet normalization and data augmentation.

In [ ]:
train_transform = transforms.Compose([
   transforms.RandomResizedCrop(

        224,

        scale=(0.8, 1.0)

    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])



test_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [ ]:
dataset_path = "../dataset"

dataset = datasets.ImageFolder(dataset_path)

labels = dataset.targets

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

## Training Utilities

Define reusable functions for model training and validation.

In [ ]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
def evaluate(model, loader):

    model.eval()

    predictions = []
    targets = []

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            predictions.extend(preds.cpu().numpy())
            targets.extend(labels.cpu().numpy())

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    accuracy = correct / total

    return accuracy, predictions, targets

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Train ResNet18

Fine-tune a pretrained ResNet18 model using transfer learning.

The experiment includes:

- Frozen feature extractor
- Fine-tuned classification layer
- AdamW optimizer
- Early stopping
- Learning-rate scheduling

In [ ]:
fold_results = []

best_fold_accuracy = 0
best_fold = -1

os.makedirs("../models", exist_ok=True)

EPOCHS = 40
PATIENCE = 5

for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels), 1):

    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    train_dataset = copy.deepcopy(dataset)
    train_dataset.transform = train_transform

    val_dataset = copy.deepcopy(dataset)
    val_dataset.transform = test_transform

    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(val_dataset, val_idx)

    train_loader = DataLoader(
        train_subset,
        batch_size=8,
        shuffle=True
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=8,
        shuffle=False
    )

    # --------------------------------------------------
    # Model
    # --------------------------------------------------

    weights = models.ResNet18_Weights.DEFAULT

    model = models.resnet18(weights=weights)

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Fine-tune last TWO residual blocks
    for param in model.layer3.parameters():
        param.requires_grad = True

    for param in model.layer4.parameters():
        param.requires_grad = True

    # Replace classifier
    model.fc = nn.Linear(512, 2)

    # Train classifier
    for param in model.fc.parameters():
        param.requires_grad = True

    model = model.to(device)

    # --------------------------------------------------
    # Loss
    # --------------------------------------------------

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )

    # --------------------------------------------------
    # Optimizer
    # --------------------------------------------------

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=5e-5,
        weight_decay=1e-4
    )

    # --------------------------------------------------
    # Scheduler
    # --------------------------------------------------

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    # --------------------------------------------------
    # Early Stopping
    # --------------------------------------------------

    best_acc = 0
    counter = 0

    best_model = copy.deepcopy(model.state_dict())

    # --------------------------------------------------
    # Training
    # --------------------------------------------------

    for epoch in range(EPOCHS):

        loss, train_acc = train_one_epoch(model, train_loader)

        val_acc, preds, targets = evaluate(model, val_loader)

        scheduler.step(val_acc)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {loss:.4f} | "
            f"Train: {train_acc:.4f} | "
            f"Val: {val_acc:.4f}"
        )

        if val_acc > best_acc:

            best_acc = val_acc
            best_model = copy.deepcopy(model.state_dict())
            counter = 0

        else:

            counter += 1

        if counter >= PATIENCE:

            print("Early Stopping...")
            break

    # --------------------------------------------------
    # Load Best Model
    # --------------------------------------------------

    model.load_state_dict(best_model)

    val_acc, preds, targets = evaluate(model, val_loader)

    precision = precision_score(targets, preds)

    recall = recall_score(targets, preds)

    f1 = f1_score(targets, preds)

    print(f"\nBest Validation Accuracy : {val_acc:.4f}")

    # --------------------------------------------------
    # Save Best Fold
    # --------------------------------------------------

    if val_acc > best_fold_accuracy:

        best_fold_accuracy = val_acc
        best_fold = fold

        torch.save(
            model.state_dict(),
            "../models/best_resnet18_detector.pth"
        )

        print("Best model saved!")

    fold_results.append({

        "accuracy": val_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1

    })



In [ ]:
# ---------------- Results ---------------- #

acc = [x["accuracy"] for x in fold_results]
prec = [x["precision"] for x in fold_results]
rec = [x["recall"] for x in fold_results]
f1 = [x["f1"] for x in fold_results]

print("\n" + "=" * 60)
print("Cross Validation Results")
print("=" * 60)

print("Accuracy :", acc)
print()
print("Precision :", prec)
print()
print("Recall :", rec)
print()
print("F1 :", f1)
print()

print(f"Mean Accuracy : {np.mean(acc):.4f}")
print(f"Std Accuracy  : {np.std(acc):.4f}")
print()

print(f"Mean Precision : {np.mean(prec):.4f}")
print(f"Mean Recall    : {np.mean(rec):.4f}")
print(f"Mean F1 Score  : {np.mean(f1):.4f}")

print("\n" + "=" * 60)
print(f"Best Fold : {best_fold}")
print(f"Best Accuracy : {best_fold_accuracy:.4f}")
print("Best model saved as:")
print("=" * 60)

In [ ]:
print(dataset.class_to_idx)

# Summary

This notebook evaluated ResNet18 using Stratified 5-Fold Cross Validation.

The experiment demonstrated that transfer learning significantly outperformed the previously developed classical machine learning models while maintaining relatively low computational cost.

The results also served as a baseline for comparison with deeper architectures such as ResNet34.